# moe_trading_agent — Colab Notebook

Mixture of Experts (MoE) transformer for 3-class trading prediction.
Each MoE transformer layer has 8 experts with top-2 sparse routing plus an
auxiliary load-balancing loss.

Self-contained. Recommended runtime: **T4 GPU** (you can try A100/L4 in Colab
Pro for bigger experts). Run all cells top-to-bottom.


In [ ]:
!pip install -q pyarrow polars pyyaml scikit-learn pandas
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 1. Tokenizers (delta + bucket)


In [ ]:
"""Shared tokenizers (copied from token_first_transformer/tokenizer/)."""
from __future__ import annotations

from pathlib import Path

import numpy as np


class DeltaTokenizer:
    """Percentage-delta tokenizer (0=PAD, 1=CLS, 2+=bins)."""

    def __init__(self, range_pct: float = 3.0, step_pct: float = 0.05) -> None:
        self.range_pct = range_pct
        self.step_pct = step_pct
        self.n_bins = int(2 * range_pct / step_pct)
        self.pad_id = 0
        self.cls_id = 1
        self._offset = 2
        self.vocab_size = self._offset + self.n_bins

    def encode_batch(self, deltas: np.ndarray) -> np.ndarray:
        deltas_pct = deltas * 100
        bin_idx = np.round(deltas_pct / self.step_pct).astype(np.int32) + self.n_bins // 2 - 1
        bin_idx = np.clip(bin_idx, 0, self.n_bins - 1)
        return (bin_idx + self._offset).astype(np.int32)

    def from_closes(self, closes: np.ndarray) -> np.ndarray:
        n = len(closes)
        ids = np.full(n, self.pad_id, dtype=np.int32)
        if n < 2:
            return ids
        deltas = (closes[1:] - closes[:-1]) / closes[:-1]
        ids[1:] = self.encode_batch(deltas)
        return ids


class BucketTokenizer:
    """Quantile bucket tokenizer (0=PAD, 1=reserved CLS, 2+=bins)."""

    def __init__(self, n_bins: int = 8) -> None:
        self.n_bins = n_bins
        self.pad_id = 0
        self._offset = 2
        self.vocab_size = self._offset + n_bins
        self.boundaries = None

    def fit(self, values: np.ndarray) -> None:
        quantiles = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, quantiles).astype(np.float32)

    def encode_batch(self, values: np.ndarray) -> np.ndarray:
        assert self.boundaries is not None
        bin_idx = np.searchsorted(self.boundaries, values, side="right")
        return (bin_idx + self._offset).astype(np.int32)


## 2. Indicator computer


In [ ]:
"""Technical-indicator computer (from indicator_tokenizer/indicators/computer.py)."""
import numpy as np


def _ema(arr, span):
    alpha = 2.0 / (span + 1)
    out = np.zeros(len(arr), dtype=np.float64)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = alpha * arr[i] + (1 - alpha) * out[i - 1]
    return out


class IndicatorComputer:
    def rsi(self, close, period=14):
        delta = np.diff(close.astype(np.float64), prepend=close[0])
        gain = np.where(delta > 0, delta, 0.0)
        loss = np.where(delta < 0, -delta, 0.0)
        avg_gain = _ema(gain, period)
        avg_loss = _ema(loss, period)
        rs = avg_gain / (avg_loss + 1e-10)
        return (100 - 100 / (1 + rs)).astype(np.float32)

    def macd_hist(self, close, fast=12, slow=26, signal=9):
        c = close.astype(np.float64)
        macd_line = _ema(c, fast) - _ema(c, slow)
        signal_line = _ema(macd_line, signal)
        return (macd_line - signal_line).astype(np.float32)

    def bollinger_pctb(self, close, period=20, num_std=2.0):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            w = c[i - period + 1 : i + 1]
            sma = w.mean(); std = w.std()
            upper = sma + num_std * std; lower = sma - num_std * std
            out[i] = (c[i] - lower) / (upper - lower + 1e-10)
        return out

    def atr(self, high, low, close, period=14):
        tr = np.zeros(len(close), dtype=np.float64)
        tr[0] = high[0] - low[0]
        tr[1:] = np.maximum(
            high[1:] - low[1:],
            np.maximum(
                np.abs(high[1:].astype(np.float64) - close[:-1].astype(np.float64)),
                np.abs(low[1:].astype(np.float64) - close[:-1].astype(np.float64)),
            ),
        )
        return _ema(tr, period).astype(np.float32)

    def volume_ratio(self, close, volume, period=20):
        v = volume.astype(np.float64)
        out = np.zeros(len(v), dtype=np.float32)
        for i in range(period - 1, len(v)):
            sma = v[i - period + 1 : i + 1].mean()
            out[i] = v[i] / (sma + 1e-10)
        return out

    def price_vs_sma(self, close, period=20):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            sma = c[i - period + 1 : i + 1].mean()
            out[i] = (c[i] - sma) / (sma + 1e-10)
        return out

    def compute_all(self, ohlcv):
        return {
            "rsi": self.rsi(ohlcv["close"]),
            "macd_hist": self.macd_hist(ohlcv["close"]),
            "bollinger_pctb": self.bollinger_pctb(ohlcv["close"]),
            "atr": self.atr(ohlcv["high"], ohlcv["low"], ohlcv["close"]),
            "volume_ratio": self.volume_ratio(ohlcv["close"], ohlcv["volume"]),
            "price_vs_sma": self.price_vs_sma(ohlcv["close"]),
        }


## 3. Indicator tokenizer


In [ ]:
"""Indicator tokenizer (from indicator_tokenizer/indicators/tokenizer.py)."""
from pathlib import Path
import numpy as np


class FixedBoundaries:
    def __init__(self, bins, offset=2):
        self.bins = np.array(bins, dtype=np.float32)
        self.offset = offset
        self.vocab_size = len(bins) + 1 + offset

    def encode_batch(self, values):
        return (np.searchsorted(self.bins, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.bins)
    def load(self, path): self.bins = np.load(path)


class QuantileBoundaries:
    def __init__(self, n_bins, offset=2):
        self.n_bins = n_bins
        self.offset = offset
        self.vocab_size = n_bins + offset
        self.boundaries = None

    def fit(self, values):
        q = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, q).astype(np.float32)

    def encode_batch(self, values):
        assert self.boundaries is not None
        return (np.searchsorted(self.boundaries, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.boundaries)
    def load(self, path): self.boundaries = np.load(path)


class IndicatorTokenizer:
    PAD_ID = 0; SPECIAL_ID = 1

    def __init__(self):
        self.rsi = FixedBoundaries(bins=[20, 30, 70, 80])
        self.macd_hist = QuantileBoundaries(n_bins=7)
        self.bollinger_pctb = FixedBoundaries(bins=[0.0, 0.25, 0.75, 1.0])
        self.atr = QuantileBoundaries(n_bins=6)
        self.volume_ratio = QuantileBoundaries(n_bins=5)
        self.price_vs_sma = QuantileBoundaries(n_bins=5)
        self._quantile_fields = ["macd_hist", "atr", "volume_ratio", "price_vs_sma"]
        self._all_fields = ["rsi", "macd_hist", "bollinger_pctb", "atr",
                            "volume_ratio", "price_vs_sma"]

    def fit(self, ind):
        for f in self._quantile_fields:
            getattr(self, f).fit(ind[f])

    def encode(self, ind):
        return {f: getattr(self, f).encode_batch(ind[f]) for f in self._all_fields}

    def vocab_sizes(self):
        return {f: getattr(self, f).vocab_size for f in self._all_fields}

    def save(self, d):
        d = Path(d); d.mkdir(parents=True, exist_ok=True)
        for f in self._all_fields:
            getattr(self, f).save(d / f"{f}.npy")

    def load(self, d):
        d = Path(d)
        for f in self._all_fields:
            getattr(self, f).load(d / f"{f}.npy")


## 4. Router, Expert, MoELayer


In [ ]:
"""Top-K routing with load-balancing aux loss."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class Router(nn.Module):
    def __init__(self, dim, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(dim, num_experts)

    def forward(self, x):
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = torch.topk(probs, self.top_k, dim=-1)
        top_w = top_w / (top_w.sum(dim=-1, keepdim=True) + 1e-10)
        one_hot = F.one_hot(top_i, num_classes=self.num_experts).float()
        mask = one_hot.sum(dim=1)
        f = mask.mean(dim=0); p = probs.mean(dim=0)
        aux = self.num_experts * torch.sum(f * p)
        return top_i, top_w, aux


class Expert(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden_dim), nn.ReLU(),
                                 nn.Linear(hidden_dim, dim))

    def forward(self, x): return self.net(x)


class MoELayer(nn.Module):
    def __init__(self, dim, hidden_dim, num_experts=8, top_k=2):
        super().__init__()
        self.num_experts = num_experts; self.top_k = top_k
        self.experts = nn.ModuleList([Expert(dim, hidden_dim) for _ in range(num_experts)])
        self.router = Router(dim, num_experts, top_k)

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)
        top_i, top_w, aux = self.router(flat)
        out = torch.zeros_like(flat)
        for k in range(self.top_k):
            exp_idx = top_i[:, k]; w = top_w[:, k]
            for e in range(self.num_experts):
                mask = (exp_idx == e)
                if not mask.any(): continue
                ei = flat[mask]
                eo = self.experts[e](ei)
                out[mask] += w[mask].unsqueeze(-1) * eo
        return out.reshape(B, T, D), aux


## 5. MoE model


In [ ]:
"""MoETradingModel (stacked MoE-transformer blocks)."""
import torch
import torch.nn as nn


class MoETransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim, num_experts, top_k, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.moe = MoELayer(dim, hidden_dim, num_experts, top_k)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        r = x
        n = self.norm1(x)
        a, _ = self.attn(n, n, n)
        x = r + self.dropout(a)
        r = x
        n = self.norm2(x)
        m, aux = self.moe(n)
        x = r + self.dropout(m)
        return x, aux


class MoETradingModel(nn.Module):
    def __init__(self, seq_len=128, num_experts=8, top_k=2, num_layers=4,
                 num_heads=8, dim=256, hidden_dim=1024, dropout=0.1, num_classes=3):
        super().__init__()
        self.seq_len = seq_len; self.dim = dim
        self.delta_emb = nn.Embedding(122, 64)
        self.vol_emb = nn.Embedding(10, 16)
        self.vb_emb = nn.Embedding(10, 16)
        self.candle_proj = nn.Linear(96, 128)
        self.indicator_embs = nn.ModuleList([
            nn.Embedding(7, 16), nn.Embedding(9, 16), nn.Embedding(7, 16),
            nn.Embedding(8, 16), nn.Embedding(7, 16), nn.Embedding(7, 16),
        ])
        self.indicator_proj = nn.Linear(96, 128)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_emb = nn.Embedding(seq_len + 1, dim)
        self.layers = nn.ModuleList([
            MoETransformerBlock(dim, num_heads, hidden_dim, num_experts, top_k, dropout)
            for _ in range(num_layers)])
        self.head = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, delta_ids, vol_ids, vb_ids, ind_dict):
        B = delta_ids.shape[0]
        de = self.delta_emb(delta_ids); ve = self.vol_emb(vol_ids); vbe = self.vb_emb(vb_ids)
        candle = self.candle_proj(torch.cat([de, ve, vbe], dim=-1))
        keys = ["rsi","macd_hist","bollinger_pctb","atr","volume_ratio","price_vs_sma"]
        ie = [emb(ind_dict[k]) for emb, k in zip(self.indicator_embs, keys)]
        indicator = self.indicator_proj(torch.cat(ie, dim=-1))
        x = torch.cat([candle, indicator], dim=-1)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        pos = torch.arange(x.shape[1], device=x.device)
        x = x + self.pos_emb(pos).unsqueeze(0)
        aux_total = torch.tensor(0.0, device=x.device)
        for layer in self.layers:
            x, aux = layer(x)
            aux_total = aux_total + aux
        return self.head(x[:, 0]), aux_total


## 6. MoEDataset


In [ ]:
"""MoEDataset (from moe_trading_agent/dataset/moe_dataset.py, tokenizers inlined)."""
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class MoEDataset(Dataset):
    def __init__(self, file_paths, seq_len=128, horizon=60, threshold=0.0015,
                 boundaries_dir=None):
        self.seq_len = seq_len; self.horizon = horizon; self.threshold = threshold

        dfs = [pd.read_parquet(fp) for fp in file_paths]
        data = pd.concat(dfs, ignore_index=True)
        self.open = data["open"].values.astype(np.float32)
        self.high = data["high"].values.astype(np.float32)
        self.low = data["low"].values.astype(np.float32)
        self.close = data["close"].values.astype(np.float32)
        self.volume = data["volume"].values.astype(np.float32)

        self.delta_tokenizer = DeltaTokenizer(range_pct=3.0, step_pct=0.05)
        self.vol_tokenizer = BucketTokenizer(n_bins=8); self.vol_tokenizer.fit(self.volume)
        vb_raw = np.abs(self.close - self.open) / (self.high - self.low + 1e-10)
        self.vb_tokenizer = BucketTokenizer(n_bins=8); self.vb_tokenizer.fit(vb_raw)

        self.comp = IndicatorComputer()
        ohlcv = {"open": self.open, "high": self.high, "low": self.low,
                 "close": self.close, "volume": self.volume}
        self.indicators_raw = self.comp.compute_all(ohlcv)

        self.ind_tok = IndicatorTokenizer()
        if boundaries_dir is not None and (Path(boundaries_dir) / "rsi.npy").exists():
            self.ind_tok.load(boundaries_dir)
        else:
            self.ind_tok.fit(self.indicators_raw)
        self.indicators_tokenized = self.ind_tok.encode(self.indicators_raw)

        self.delta_ids_all = self.delta_tokenizer.from_closes(self.close)
        self.vol_ids_all = self.vol_tokenizer.encode_batch(self.volume)
        self.vb_ids_all = self.vb_tokenizer.encode_batch(vb_raw)

    def __len__(self):
        return len(self.close) - self.seq_len - self.horizon

    def __getitem__(self, idx):
        s, e = idx, idx + self.seq_len
        delta = torch.tensor(self.delta_ids_all[s:e], dtype=torch.long)
        vol = torch.tensor(self.vol_ids_all[s:e], dtype=torch.long)
        vb = torch.tensor(self.vb_ids_all[s:e], dtype=torch.long)
        ind = {k: torch.tensor(self.indicators_tokenized[k][s:e], dtype=torch.long)
               for k in self.indicators_tokenized}
        future = self.close[e + self.horizon]
        current = self.close[e - 1]
        p = (future - current) / (current + 1e-10)
        if p > self.threshold: label = 0
        elif p < -self.threshold: label = 2
        else: label = 1
        return delta, vol, vb, ind, torch.tensor(label, dtype=torch.long)


## 7. MoE trainer


In [ ]:
"""Trainer for MoETradingModel."""
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score


def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")


class MoETrainer:
    def __init__(self, model, train_loader, val_loader, device=None,
                 lr=3e-4, weight_decay=0.01, aux_loss_lambda=0.01,
                 patience=3, checkpoint_dir="/content/checkpoints"):
        self.device = device or get_device()
        self.model = model.to(self.device)
        self.train_loader = train_loader; self.val_loader = val_loader
        self.aux_lambda = aux_loss_lambda; self.patience = patience
        self.ckpt_dir = Path(checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.class_weights = self._compute_class_weights().to(self.device)
        self.criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        self.opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        self.best_f1 = 0.0; self.pat = 0

    def _compute_class_weights(self):
        counts = torch.zeros(3)
        for b in self.train_loader:
            y = b[-1]
            for c in range(3):
                counts[c] += (y == c).sum().item()
        w = 1.0 / (counts + 1e-6)
        return (w / w.sum() * 3).float()

    def _train_ep(self):
        self.model.train(mode=True); tl, n = 0.0, 0
        for b in self.train_loader:
            d, vo, vb, ind, y = b
            d = d.to(self.device); vo = vo.to(self.device); vb = vb.to(self.device)
            ind = {k: v.to(self.device) for k, v in ind.items()}
            y = y.to(self.device)
            logits, aux = self.model(d, vo, vb, ind)
            loss = self.criterion(logits, y) + self.aux_lambda * aux
            self.opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
            tl += loss.item(); n += 1
        return tl / max(n, 1)

    def _val_ep(self):
        self.model.train(mode=False)
        tl, n = 0.0, 0; preds, labels = [], []
        with torch.no_grad():
            for b in self.val_loader:
                d, vo, vb, ind, y = b
                d = d.to(self.device); vo = vo.to(self.device); vb = vb.to(self.device)
                ind = {k: v.to(self.device) for k, v in ind.items()}
                y = y.to(self.device)
                logits, aux = self.model(d, vo, vb, ind)
                loss = self.criterion(logits, y) + self.aux_lambda * aux
                tl += loss.item(); n += 1
                preds.extend(logits.argmax(-1).cpu().numpy())
                labels.extend(y.cpu().numpy())
        return tl / max(n, 1), float(f1_score(labels, preds, average="weighted", zero_division=0))

    def train(self, epochs, use_scheduler=True):
        sched = (torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=epochs)
                 if use_scheduler else None)
        print(f"device={self.device}  class_weights={self.class_weights.cpu().numpy()}")
        for ep in range(1, epochs + 1):
            tl = self._train_ep(); vl, vf = self._val_ep()
            if sched is not None: sched.step()
            print(f"Ep {ep}/{epochs}: train={tl:.4f} val={vl:.4f} f1={vf:.4f}")
            if vf > self.best_f1:
                self.best_f1 = vf; self.pat = 0
                torch.save({"model_state_dict": self.model.state_dict(), "val_f1": vf, "epoch": ep},
                           self.ckpt_dir / "best_model.pt")
                print(f"  -> saved (f1={vf:.4f})")
            else:
                self.pat += 1
                if self.pat >= self.patience:
                    print(f"  -> early stop ep{ep}"); break
        print(f"best f1={self.best_f1:.4f}")


## 8. Configuration + data paths


In [ ]:
"""Configuration."""
from pathlib import Path

USE_MOCK_DATA = True
if USE_MOCK_DATA:
    DATA_DIR = Path("/content/mock_data")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/trading_data")

MOCK_MONTHS = ["2024-01", "2024-02", "2024-03"]
MOCK_MINUTES_PER_MONTH = 30000
SYMBOL = "BTCUSDT"

MODEL_CFG = {
    "seq_len": 128, "num_experts": 8, "top_k": 2, "num_layers": 4,
    "num_heads": 8, "dim": 256, "hidden_dim": 1024,
    "dropout": 0.1, "num_classes": 3,
}
TRAIN_CFG = {
    "batch_size": 32, "lr": 3e-4, "weight_decay": 0.01,
    "aux_loss_lambda": 0.01, "epochs": 5, "patience": 3,
    "horizon": 60, "threshold": 0.0015,
    "train_split": 0.8, "val_split": 0.1,
    "checkpoint_dir": "/content/checkpoints",
}

if Path("/content/drive/MyDrive").exists():
    ARTIFACTS_ROOT = Path("/content/drive/MyDrive/w_training/moe_trading_agent")
else:
    ARTIFACTS_ROOT = Path("/content/artifacts/moe_trading_agent")
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)


## 9. Mock data generation


In [ ]:
"""Generate synthetic BTCUSDT 1-min OHLCV (geometric Brownian motion).
Skipped automatically when USE_MOCK_DATA = False."""
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


def _gen_month(n_minutes, start_price, seed):
    rng = np.random.default_rng(seed)
    sigma = 0.0008
    lr = rng.normal(0.0, sigma, size=n_minutes)
    closes = start_price * np.exp(np.cumsum(lr))
    opens = np.concatenate([[start_price], closes[:-1]])
    noise = np.abs(rng.normal(0, sigma, size=n_minutes)) * closes
    highs = np.maximum(opens, closes) + noise
    lows = np.minimum(opens, closes) - noise
    vols = rng.lognormal(3.0, 1.0, size=n_minutes).astype(np.float32)
    return {
        "open": opens.astype(np.float32),
        "high": highs.astype(np.float32),
        "low": lows.astype(np.float32),
        "close": closes.astype(np.float32),
        "volume": vols,
    }


if USE_MOCK_DATA:
    kd = DATA_DIR / "BTCUSDT" / "klines_1m"
    kd.mkdir(parents=True, exist_ok=True)
    start_price = 42000.0
    for i, m in enumerate(MOCK_MONTHS):
        out = kd / f"{m}.parquet"
        if out.exists():
            print("skip", out.name); continue
        d = _gen_month(MOCK_MINUTES_PER_MONTH, start_price, 42 + i)
        start_price = float(d["close"][-1])
        pq.write_table(pa.table(d), out)
        print(f"wrote {out.name}  rows={MOCK_MINUTES_PER_MONTH}")
    print("files:", sorted(p.name for p in kd.glob("*.parquet")))
else:
    print("USE_MOCK_DATA=False — expecting real data at", DATA_DIR)


## 10. Main


In [ ]:
"""Main: gather parquet files, build dataset & splits, train MoE, evaluate."""
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

klines_dir = DATA_DIR / SYMBOL / "klines_1m"
files = sorted([str(f) for f in klines_dir.glob("*.parquet")])
if not files:
    raise SystemExit(f"no parquet files in {klines_dir}")
print(f"found {len(files)} parquet files")

ds = MoEDataset(files, seq_len=MODEL_CFG["seq_len"],
                horizon=TRAIN_CFG["horizon"], threshold=TRAIN_CFG["threshold"])
print(f"dataset size: {len(ds)}")

total = len(ds)
tr_end = int(total * TRAIN_CFG["train_split"])
va_end = int(total * (TRAIN_CFG["train_split"] + TRAIN_CFG["val_split"]))
train_ds = Subset(ds, range(0, tr_end))
val_ds   = Subset(ds, range(tr_end, va_end))
test_ds  = Subset(ds, range(va_end, total))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

tr_dl = DataLoader(train_ds, batch_size=TRAIN_CFG["batch_size"], shuffle=True,  num_workers=0)
va_dl = DataLoader(val_ds,   batch_size=TRAIN_CFG["batch_size"], shuffle=False, num_workers=0)
te_dl = DataLoader(test_ds,  batch_size=TRAIN_CFG["batch_size"], shuffle=False, num_workers=0)

model = MoETradingModel(**MODEL_CFG)
print(f"params={sum(p.numel() for p in model.parameters()):,}")

trainer = MoETrainer(model, tr_dl, va_dl,
    lr=TRAIN_CFG["lr"], weight_decay=TRAIN_CFG["weight_decay"],
    aux_loss_lambda=TRAIN_CFG["aux_loss_lambda"], patience=TRAIN_CFG["patience"],
    checkpoint_dir=TRAIN_CFG["checkpoint_dir"])
trainer.train(epochs=TRAIN_CFG["epochs"])

print("\n=== Test evaluation ===")
dev = trainer.device
ckpt = Path(TRAIN_CFG["checkpoint_dir"]) / "best_model.pt"
state = torch.load(ckpt, map_location=dev, weights_only=False)
model.load_state_dict(state["model_state_dict"]); model.train(mode=False)
all_preds, all_labels = [], []
with torch.no_grad():
    for b in te_dl:
        d, vo, vb, ind, y = b
        d = d.to(dev); vo = vo.to(dev); vb = vb.to(dev)
        ind = {k: v.to(dev) for k, v in ind.items()}
        logits, _ = model(d, vo, vb, ind)
        all_preds.extend(logits.argmax(-1).cpu().numpy().tolist())
        all_labels.extend(y.numpy().tolist())
print(classification_report(all_labels, all_preds,
    target_names=["UP","FLAT","DOWN"], zero_division=0))


## 11. Save artifacts to Google Drive
Persists the MoE checkpoint, fitted tokenizers, config and test metrics to `ARTIFACTS_ROOT` (Drive or `/content/artifacts`).


In [ ]:
"""Save checkpoint, tokenizers, metrics and config to ARTIFACTS_ROOT."""
import json
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACTS_ROOT / f"run_{RUN_TAG}"
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "tokenizers").mkdir(parents=True, exist_ok=True)

src = Path(TRAIN_CFG["checkpoint_dir"]) / "best_model.pt"
if src.exists():
    shutil.copy(src, RUN_DIR / "checkpoints" / "best_model.pt")

np.save(RUN_DIR / "tokenizers" / "vol_boundaries.npy", ds.vol_tokenizer.boundaries)
np.save(RUN_DIR / "tokenizers" / "vb_boundaries.npy",  ds.vb_tokenizer.boundaries)
with open(RUN_DIR / "tokenizers" / "delta_params.json", "w") as f:
    json.dump({
        "range_pct": ds.delta_tokenizer.range_pct,
        "step_pct":  ds.delta_tokenizer.step_pct,
    }, f, indent=2)
ds.ind_tok.save(RUN_DIR / "tokenizers" / "indicators")

with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"model": MODEL_CFG, "training": TRAIN_CFG,
               "symbol": SYMBOL, "data_dir": str(DATA_DIR)},
              f, indent=2, default=str)

test_report_dict = classification_report(
    all_labels, all_preds, target_names=["UP","FLAT","DOWN"],
    zero_division=0, output_dict=True)
with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "report":           test_report_dict,
        "confusion_matrix": confusion_matrix(all_labels, all_preds).tolist(),
    }, f, indent=2)

np.savez_compressed(
    RUN_DIR / "predictions.npz",
    preds=np.asarray(all_preds, dtype=np.int8),
    labels=np.asarray(all_labels, dtype=np.int8),
)

print(f"saved to: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)!s:40s}  {p.stat().st_size:>10,} B")
